In [4]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, MessagesState
from langgraph.graph import START, END
from llm_factory import LLMFactory
from langchain_core.messages import SystemMessage, trim_messages
from typing import TypedDict  

In [5]:
from config import DATA_DIR
files = [
    DATA_DIR / "file_03.csv",
]

print(files)
data = []
with open(files[0], 'r') as f:
    for line in f:
        data.append(line)
print(type(data))
print(f"Total de eventos: {len(data)}")
print(f"Exemplo:\n {data[1]}")


[WindowsPath('C:/davi_tonon/mestrado/data/raw/file_03.csv')]
<class 'list'>
Total de eventos: 302
Exemplo:
 AwsApiCall,ListObjects,E2E91AQVN6DJHZ91,"{'type': 'AWSAccount', 'principalId': '', 'accountId': 'ANONYMOUS_PRINCIPAL'}","{'bucketName': 'microsoft-devtest', 'Host': 'microsoft-devtest.s3-eu-west-1.amazonaws.com'}", 302-2022-02-18T17:34:57Z,283770f5-968d-448d-9328-0b010f4d3696,2022-02-18T17:34:57Z,2022-02-18T17:39:28.127636+00:00,177.131.167.145,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/98.0.4758.102 Safari/537.36",4



In [6]:
SYSTEM_PROMPT = """
You are a cybersecurity analyst specialized in log analysis and MITRE ATT&CK mapping.
You will receive many event of a file. Analyze each event carefully.

You are able to:
- Detect anomalies in logs
- Identify security events
- Map findings to MITRE ATT&CK TTPs

Rules:
- Always base your analysis on evidence
- Do NOT hallucinate
- Be precise and technical
- Always answer in English

### OUTPUT REQUIREMENTS
Provide the results in the following structured format:

**MITRE ATT&CK Mapping**
- **Tactics:**
- **Techniques:**
- **Technique IDs:**



"""

In [7]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState  # ← Importante!
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# ❌ NÃO use TypedDict customizado para messages
# ✅ Use MessagesState que já sabe acumular

def analyze_logs(state: MessagesState):
    llm = LLMFactory().get_model()
    llm = ChatOllama(model="llama3.1:8b", temperature=0.1)
    
    # MessagesState já tem 'messages' como chave
    response = llm.invoke(state["messages"])
    
    # Retorna a nova mensagem (MessagesState acumula automaticamente)
    return {"messages": [response]}

builder = StateGraph(MessagesState)  # ← MessagesState aqui
builder.add_node("analyze", analyze_logs)
builder.add_edge(START, "analyze")
builder.add_edge("analyze", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "sessao-teste"}}

# Teste
result1 = graph.invoke(
    {"messages": [HumanMessage(content=SYSTEM_PROMPT)]},
    config
)
print("Resposta 1:", result1["messages"][-1].content)

for i in range(0, len(data), 10):
    chunk = data[i:i+10]
    result2 = graph.invoke(
        {"messages": [HumanMessage(content=f"Analyze this log :\n{chunk}?")]},
        config
    )
    print(f"Resposta {i}:", result2["messages"][-1].content)

# Verifica estado
estado = graph.get_state(config)
print(f"\nTotal de mensagens: {len(estado.values['messages'])}")
for msg in estado.values['messages']:
    print(f"  [{msg.type}] {msg.content[:80]}...")

LLMFactory: Getting model for provider 'ollama' - metis
Resposta 1: I'm ready to analyze the events. Please provide the log data, and I'll carefully examine each event, detect anomalies, identify security events, and map my findings to MITRE ATT&CK TTPs.

Please go ahead and share the logs. I'll structure my analysis according to the requirements you specified.
LLMFactory: Getting model for provider 'ollama' - metis
Resposta 0: After analyzing the log data, I have identified some potential security events and anomalies.

**MITRE ATT&CK Mapping**

* **Tactics:**
	+ Reconnaissance
	+ Lateral Movement
	+ Command and Control
* **Techniques:**
	+ T1041 (Network Device Misconfiguration)
	+ T1055 (Process Injection)
	+ T1071 (Application Layer Protocol)
	+ T1102 (Remote File Copy)
	+ T1204 (User Execution)
* **Technique IDs:**

Here's a breakdown of my analysis:

1. The log data appears to be related to AWS API calls, specifically ListObjects and HeadBucket operations.
2. There are multiple i

In [8]:
result1 = graph.invoke(
    {"messages": [HumanMessage(content='Provide a final consolidated analysis of all logs and mapping for Mitre ATT&CK')]},
    config
)
print("Resposta 1:", result1["messages"][-1].content)

LLMFactory: Getting model for provider 'ollama' - metis
Resposta 1: **Consolidated Analysis:**

Based on the provided log data, we have identified several potential threats and anomalies. Here is a consolidated summary:

* **TTPs (Tactics, Techniques, and Procedures):**
	+ Tactic: Reconnaissance
	+ Technique: ListObjects ( AWS S3 )
	+ Procedure: Multiple requests made from different IP addresses and user agents to enumerate metadata about the bucket's contents.
* **IP Addresses:**
	+ 34.71.42.209 ( multiple requests )
	+ 43.251.92.37
	+ 95.217.6.207
	+ 34.68.153.199 ( two consecutive requests )
* **User Agents:**
	+ python-requests/2.22.0
	+ aws-cli/1.17.9 Python/3.6.0 Windows/10 botocore/1.14.9
	+ Go-http-client/1.1
* **AWS Accounts:**
	+ 725677763773 ( valid AWS account )
	+ ANONYMOUS_PRINCIPAL ( anonymous principal )

**Mitre ATT&CK Mapping:**

The identified TTPs can be mapped to the following Mitre ATT&CK techniques:

* **Tactic:** Reconnaissance
* **Technique:** ListObjects ( AWS